<a href="https://colab.research.google.com/github/gremlin97/EVA-8/blob/main/Segmentation/MultimodalSegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install torchinfo
!pip install torchviz
!pip install git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done
  Created wheel for torchviz: filename=torchviz-0.0.2-py3-none-any.whl size=4131 sha256=0a7f867fce3527e1d86444481ead8d15ee9f204dd21e6215db8a69193901371f
  Stored in directory: /root/.cache/pip/wheels/4c/97/88/a02973217949e0db0c9f4346d154085f4725f99c4f15a87094
Successfully built torchviz
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-0r91tj00
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-0r91tj00
  Resolved https://github.com/openai/CLIP.git to commit a1d071733d7111c9c014f024669f959182114e33
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 1.6 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369497 sha256=19468ea5f588d1e8d680b9242c2fd667dc918f1ef11bfb92e4fe9d4c79a64bf7
  Stored in directory: /tmp/pip-ephem-wheel-cache-w62l_u82/wheels/da/2b/4c/d6691fa9597aac8bb85d2ac13b112deb897d5b

In [ ]:
import torch
import torchvision
from torch import nn
import clip
from transformers import DistilBertTokenizer, DistilBertModel
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
db = DistilBertModel.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

# UNet Normal

In [ ]:
class ContractingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ContractingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    skip = x
    x = self.maxpool(x)

    return skip, x

class ExpandingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ExpandingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.upsample = nn.ConvTranspose2d(out_channels, out_channels//2, kernel_size=2, stride=2)

  def forward(self, x, skip):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    x = self.upsample(x)

    x = torch.cat((x, skip), dim = 1)

    return x


class UNet(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(UNet, self).__init__()

    self.contract1 = ContractingBlock(in_channels, 64)
    self.contract2 = ContractingBlock(64, 128)
    self.contract3 = ContractingBlock(128, 256)

    self.expand1 = ExpandingBlock(256, 128)
    self.expand2 = ExpandingBlock(128, 64)
    self.expand3 = ExpandingBlock(64, 32)

    self.mix1 = nn.Conv2d(320, 128, kernel_size=1)
    self.mix2 = nn.Conv2d(160, 64, kernel_size=1)
    self.mix3 = nn.Conv2d(80, 64, kernel_size=1)

    self.convf = nn.Conv2d(64, out_channels, kernel_size = 1)

  def forward(self, x):
    # Contracting path
    skip1, x = self.contract1(x)
    skip2, x = self.contract2(x)
    skip3, x = self.contract3(x)

    # Expanding path
    x = self.expand1(x, skip3)
    x = self.mix1(x)
    x = self.expand2(x, skip2)
    x = self.mix2(x)
    x = self.expand3(x, skip1)
    x = self.mix3(x)
    x = self.convf(x)
    return x

In [ ]:
model_normal = UNet(1,1)
out_normal = model_normal(torch.randn(5,1,256,256))
print(out_normal.shape)

torch.Size([5, 1, 256, 256])


In [ ]:
class ContractingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ContractingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    skip = x
    x = self.maxpool(x)

    return skip, x

In [ ]:
class ExpandingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ExpandingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.upsample = nn.ConvTranspose2d(out_channels, out_channels//2, kernel_size=2, stride=2)

  def forward(self, x, skip):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    x = self.upsample(x)

    x = torch.cat((x, skip), dim = 1)

    return x

In [ ]:
class ContractingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ContractingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    skip = x
    x = self.maxpool(x)

    return skip, x


class ExpandingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ExpandingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.upsample = nn.ConvTranspose2d(out_channels, out_channels//2, kernel_size=2, stride=2)

  def forward(self, x, skip):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    x = self.upsample(x)

    x = torch.cat((x, skip), dim = 1)

    return x


class UNet_MM_BS(nn.Module):
  def __init__(self, in_channels, out_channels, tokenizer, db):
    super(UNet_MM_BS, self).__init__()

    self.contract1 = ContractingBlock(in_channels, 64)
    self.contract2 = ContractingBlock(64, 128)
    self.contract3 = ContractingBlock(128, 256)

    self.mix11 = nn.Conv2d(512, 256, kernel_size=1)
    self.mix12 = nn.Conv2d(256, 128, kernel_size=1)
    self.mix13 = nn.Conv2d(128, 64, kernel_size=1)
    self.mix14 = nn.Conv2d(128, 64, kernel_size=1)

    self.mix2 = nn.Conv2d(320, 128, kernel_size=1)
    self.mix3 = nn.Conv2d(160, 64, kernel_size=1)
    self.mix4 = nn.Conv2d(80, 64, kernel_size=1)

    self.tokenizer = tokenizer
    self.db = db

    self.linear1 = nn.Linear(768, 256)
    self.linear2 = nn.Linear(768, 128)
    self.linear3 = nn.Linear(768, 64)
    self.linear4 = nn.Linear(768, 64)

    self.expand1 = ExpandingBlock(256, 128)
    self.expand2 = ExpandingBlock(128, 64)
    self.expand3 = ExpandingBlock(64, 32)

    self.convf = nn.Conv2d(64, out_channels, kernel_size = 1)

  def forward(self, x, y):
    batch_size = x.size(0)  # Get the batch size

    print(type(y))

    skip1, x = self.contract1(x)
    skip2, x = self.contract2(x)
    skip3, x = self.contract3(x)

    embeddings = []
    for sentence in y:
        inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = db(**inputs)
            last_hidden_states = outputs.last_hidden_state
        mean_pooled = last_hidden_states.mean(dim=1)
        embeddings.append(mean_pooled)
    embeddings = torch.stack(embeddings)
    embeddings = embeddings.reshape(len(y),-1)
    y = embeddings
    base_y = y
    print(base_y.shape)

    y = self.linear1(y)
    y = y.reshape(len(y), 256 , 1, 1)
    y = y.expand(len(y), 256, 32, 32)
    x = torch.cat((x, y), dim = 1)
    x = self.mix11(x)

    y1 = self.linear2(base_y)
    y1 = y1.reshape(len(y), 128 , 1, 1)
    y1 = y1.expand(len(y), 128, 64, 64)

    y2 = self.linear3(base_y)
    y2 = y2.reshape(len(y), 64 , 1, 1)
    y2 = y2.expand(len(y), 64, 128, 128)

    y3 = self.linear4(base_y)
    y3 = y3.reshape(len(y), 64 , 1, 1)
    y3 = y3.expand(len(y), 64, 256, 256)

    x = self.expand1(x, skip3)
    x = self.mix2(x)
    x = torch.cat((x, y1), dim = 1)
    x = self.mix12(x)
    x = self.expand2(x, skip2)
    x = self.mix3(x)
    x = torch.cat((x, y2), dim = 1)
    x = self.mix13(x)
    x = self.expand3(x, skip1)
    x = self.mix4(x)
    x = torch.cat((x, y3), dim = 1)
    x = self.mix14(x)
    x = self.convf(x)
    return x

In [ ]:
model_mf = UNet_MM_BS(1,1, tokenizer, db)
out_mf = model_mf(torch.randn(5,1,256,256),['The cone is not properly segmented','The cone is well segmented','The cone is somewhat segmented','Check','Check2'])
print(out_mf.shape)

<class 'list'>
torch.Size([5, 768])
torch.Size([5, 1, 256, 256])


In [ ]:
sum(param.numel() for param in model_mf.parameters())

68765105

# 512x512 Input Compatible Model

In [ ]:
class ContractingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ContractingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    skip = x
    x = self.maxpool(x)

    return skip, x


class ExpandingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ExpandingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.upsample = nn.ConvTranspose2d(out_channels, out_channels//2, kernel_size=2, stride=2)

  def forward(self, x, skip):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    x = self.upsample(x)

    x = torch.cat((x, skip), dim = 1)

    return x


class UNet_MM_BS(nn.Module):
  def __init__(self, in_channels, out_channels, tokenizer, db):
    super(UNet_MM_BS, self).__init__()

    self.contract1 = ContractingBlock(in_channels, 64)
    self.contract2 = ContractingBlock(64, 128)
    self.contract3 = ContractingBlock(128, 256)

    self.mix11 = nn.Conv2d(512, 256, kernel_size=1)
    self.mix12 = nn.Conv2d(256, 128, kernel_size=1)
    self.mix13 = nn.Conv2d(128, 64, kernel_size=1)
    self.mix14 = nn.Conv2d(128, 64, kernel_size=1)

    self.mix2 = nn.Conv2d(320, 128, kernel_size=1)
    self.mix3 = nn.Conv2d(160, 64, kernel_size=1)
    self.mix4 = nn.Conv2d(80, 64, kernel_size=1)

    self.tokenizer = tokenizer
    self.db = db

    self.linear1 = nn.Linear(27*256, 256)
    self.linear2 = nn.Linear(27*256, 128)
    self.linear3 = nn.Linear(27*256, 64)
    self.linear4 = nn.Linear(27*256, 64)

    self.expand1 = ExpandingBlock(256, 128)
    self.expand2 = ExpandingBlock(128, 64)
    self.expand3 = ExpandingBlock(64, 32)

    self.convf = nn.Conv2d(64, out_channels, kernel_size = 1)

  def forward(self, x, y):
    batch_size = x.size(0)  # Get the batch size
    print('x',x.shape)

    skip1, x = self.contract1(x)
    skip2, x = self.contract2(x)
    skip3, x = self.contract3(x)

    # y = self.tokenizer(y, return_tensors='pt')
    y = tokenizer(y,  padding=True, truncation=True, return_tensors='pt')
    y = self.db(**y)
    y = y['last_hidden_state']
    print('Embed',y.shape)
    y = y.reshape(len(y),-1)
    base_y = y
    print('y',base_y.shape)

    y = self.linear1(y)
    y = y.reshape(len(y), 256 , 1, 1)
    y = y.expand(len(y), 256, 64, 64)

    x = torch.cat((x, y), dim = 1)
    x = self.mix11(x)

    y1 = self.linear2(base_y)
    y1 = y1.reshape(len(y), 128 , 1, 1)
    y1 = y1.expand(len(y), 128, 128, 128)

    y2 = self.linear3(base_y)
    y2 = y2.reshape(len(y), 64 , 1, 1)
    y2 = y2.expand(len(y), 64, 256, 256)

    y3 = self.linear4(base_y)
    y3 = y3.reshape(len(y), 64 , 1, 1)
    y3 = y3.expand(len(y), 64, 512, 512)

    x = self.expand1(x, skip3)
    x = self.mix2(x)
    x = torch.cat((x, y1), dim = 1)
    x = self.mix12(x)
    x = self.expand2(x, skip2)
    x = self.mix3(x)
    x = torch.cat((x, y2), dim = 1)
    x = self.mix13(x)
    x = self.expand3(x, skip1)
    x = self.mix4(x)
    x = torch.cat((x, y3), dim = 1)
    x = self.mix14(x)
    x = self.convf(x)
    return x

In [ ]:
model_mf = UNet_MM_BS(3,2, tokenizer, db)
out_mf = model_mf(torch.randn(3,3,512,512),['The cone is not properly segmented','The cone is well segmented','The cone is somewhat segmented'])
print(out_mf.shape)

x torch.Size([3, 3, 512, 512])
Embed torch.Size([3, 9, 768])
y torch.Size([3, 6912])
torch.Size([3, 2, 512, 512])


# Clip Embeddings

In [ ]:
class ContractingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ContractingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    skip = x
    x = self.maxpool(x)

    return skip, x

class ExpandingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ExpandingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.upsample = nn.ConvTranspose2d(out_channels, out_channels//2, kernel_size=2, stride=2)

  def forward(self, x, skip):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    x = self.upsample(x)

    x = torch.cat((x, skip), dim = 1)

    return x


class UNet_MM_BS(nn.Module):
  def __init__(self, in_channels, out_channels, clip_tokenizer, clip):
    super(UNet_MM_BS, self).__init__()

    self.contract1 = ContractingBlock(in_channels, 64)
    self.contract2 = ContractingBlock(64, 128)
    self.contract3 = ContractingBlock(128, 256)

    self.mix11 = nn.Conv2d(512, 256, kernel_size=1)
    self.mix12 = nn.Conv2d(256, 128, kernel_size=1)
    self.mix13 = nn.Conv2d(128, 64, kernel_size=1)
    self.mix14 = nn.Conv2d(128, 64, kernel_size=1)

    self.mix2 = nn.Conv2d(320, 128, kernel_size=1)
    self.mix3 = nn.Conv2d(160, 64, kernel_size=1)
    self.mix4 = nn.Conv2d(80, 64, kernel_size=1)

    self.clip = clip
    self.clip_tokenizer = clip_tokenizer

    self.linear1 = nn.Linear(512, 256)
    self.linear2 = nn.Linear(512, 128)
    self.linear3 = nn.Linear(512, 64)
    self.linear4 = nn.Linear(512, 64)

    self.expand1 = ExpandingBlock(256, 128)
    self.expand2 = ExpandingBlock(128, 64)
    self.expand3 = ExpandingBlock(64, 32)

    self.convf = nn.Conv2d(64, out_channels, kernel_size = 1)

  def forward(self, x, y):

    skip1, x = self.contract1(x)
    skip2, x = self.contract2(x)
    skip3, x = self.contract3(x)

    inputs = self.clip_tokenizer(y, padding=True, return_tensors="pt")
    text_features = self.clip.get_text_features(**inputs)
    y = clip.get_text_features(**inputs)
    y = y.reshape(len(y),-1)
    base_y = y
    print(base_y.shape)

    y = self.linear1(y)
    y = y.reshape(len(y), 256 , 1, 1)
    y = y.expand(len(y), 256, 64, 64)

    x = torch.cat((x, y), dim = 1)
    x = self.mix11(x)

    y1 = self.linear2(base_y)
    y1 = y1.reshape(len(y), 128 , 1, 1)
    y1 = y1.expand(len(y), 128, 128, 128)

    y2 = self.linear3(base_y)
    y2 = y2.reshape(len(y), 64 , 1, 1)
    y2 = y2.expand(len(y), 64, 256, 256)

    y3 = self.linear4(base_y)
    y3 = y3.reshape(len(y), 64 , 1, 1)
    y3 = y3.expand(len(y), 64, 512, 512)

    x = self.expand1(x, skip3)
    x = self.mix2(x)
    x = torch.cat((x, y1), dim = 1)
    x = self.mix12(x)
    x = self.expand2(x, skip2)
    x = self.mix3(x)
    x = torch.cat((x, y2), dim = 1)
    x = self.mix13(x)
    x = self.expand3(x, skip1)
    x = self.mix4(x)
    x = torch.cat((x, y3), dim = 1)
    x = self.mix14(x)
    x = self.convf(x)
    return x

In [ ]:
from transformers import AutoProcessor, AutoTokenizer, CLIPModel
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")
model_mf = UNet_MM_BS(1,1, clip_tokenizer, clip)
out_mf = model_mf(torch.randn(3,1,512,512),['The cone is not properly segmentedsssssssssssssssssssssssssssssssssssssssssssssssssssssssssss','The cone is well segmented','The cone is somewhat segmented'])
print(out_mf.shape)

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

NameError: ignored

# Multi-Modal Fusion

In [ ]:
class ContractingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ContractingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    skip = x
    x = self.maxpool(x)

    return skip, x


class ExpandingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ExpandingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.upsample = nn.ConvTranspose2d(out_channels, out_channels//2, kernel_size=2, stride=2)

  def forward(self, x, skip):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    x = self.upsample(x)

    x = torch.cat((x, skip), dim = 1)

    return x


class CrossAttention(nn.Module):
  def __init__(self, in_channels):
    super(CrossAttention, self).__init__()

    self.query = nn.Conv2d(in_channels, in_channels//2, kernel_size=1)
    self.key = nn.Conv2d(in_channels, in_channels//2, kernel_size=1)
    self.value = nn.Conv2d(in_channels, in_channels//2, kernel_size=1)

  def forward(self, x, xt):
    k = self.key(x)
    v = self.value(x)

    q = self.query(xt)

    print('s1',k.shape,v.shape,q.shape)

    q = q.view(q.size(0), -1, q.size(2) * q.size(3))
    k = k.view(k.size(0), -1, k.size(2) * k.size(3))
    v = v.view(v.size(0), -1, v.size(2) * v.size(3))

    attn_scores = F.softmax(torch.bmm(q.transpose(1, 2), k), dim=2)
    attention = torch.bmm(v, attn_scores.transpose(1, 2))
    attention = attention.view(v.size(0), v.size(1), v.size(2), xt.size(2), xt.size(3))

    return attention


class UNet_MM_BS(nn.Module):
  def __init__(self, in_channels, out_channels, clip_tokenizer, clip):
    super(UNet_MM_BS, self).__init__()

    self.contract1 = ContractingBlock(in_channels, 64)
    self.contract2 = ContractingBlock(64, 128)
    self.contract3 = ContractingBlock(128, 256)

    self.pool1 = nn.AdaptiveAvgPool2d((1,1))
    self.pool2 = nn.AdaptiveAvgPool2d((1,1))
    self.pool3 = nn.AdaptiveAvgPool2d((1,1))

    self.mix1 = nn.Conv2d(768, 256, kernel_size=1)
    self.mix2 = nn.Conv2d(640, 128, kernel_size=1)
    self.mix3 = nn.Conv2d(320, 64, kernel_size=1)
    self.mix4 = nn.Conv2d(160, 64, kernel_size=1)

    self.clip = clip
    self.clip_tokenizer = clip_tokenizer

    self.linear1 = nn.Linear(64, 512)
    self.linear2 = nn.Linear(128, 512)
    self.linear3 = nn.Linear(256, 512)
    self.linear4 = nn.Linear(512, 512)
    self.linear5 = nn.Linear(512, 512)
    self.linear6 = nn.Linear(1024, 512)
    self.linear7 = nn.Linear(1024, 512)
    self.linear8 = nn.Linear(1024, 512)
    self.linear9 = nn.Linear(1536, 512)

    self.conv1 = nn.ConvTranspose2d(in_channels=512, out_channels=320, kernel_size=3, stride=2, padding=1, output_padding=1)
    self.conv2 = nn.ConvTranspose2d(in_channels=320, out_channels=160, kernel_size=3, stride=2, padding=1, output_padding=1)
    self.conv3 = nn.ConvTranspose2d(in_channels=160, out_channels=80, kernel_size=3, stride=2, padding=1, output_padding=1)

    self.expand1 = ExpandingBlock(256, 128)
    self.expand2 = ExpandingBlock(128, 64)
    self.expand3 = ExpandingBlock(64, 32)

    self.cross = CrossAttention(in_channels)

    self.convf = nn.Conv2d(64, out_channels, kernel_size = 1)

  def forward(self, x, y):

    skip1, x = self.contract1(x)
    skip2, x = self.contract2(x)
    skip3, x = self.contract3(x)

    fuse1 = skip1
    fuse2 = skip2
    fuse3 = skip3

    fuse1 = self.pool1(fuse1)
    fuse2 = self.pool1(fuse2)
    fuse3 = self.pool1(fuse3)

    vfuse1 = fuse1.reshape(fuse1.shape[0],-1)
    vfuse2 = fuse2.reshape(fuse2.shape[0],-1)
    vfuse3 = fuse3.reshape(fuse3.shape[0],-1)

    print(vfuse1.shape, vfuse2.shape, vfuse3.shape)

    inputs = self.clip_tokenizer(y, padding=True, return_tensors="pt")
    text_features = self.clip.get_text_features(**inputs)
    y = clip.get_text_features(**inputs)
    y = y.reshape(len(y),-1)
    print(y.shape)
    base_y = y

    F3 = F.relu(self.linear3(vfuse3))*F.relu(self.linear4(base_y))
    F2 = torch.cat([F.relu(self.linear2(vfuse2)), F.relu(self.linear5(F3))], dim=1)
    F1 = torch.cat([F.relu(self.linear1(vfuse1)), F.relu(self.linear6(F2))], dim=1)
    F0 = self.linear9(torch.cat([self.linear8(F1),self.linear7(F2),F3],dim=1))

    F0 = F0.reshape(len(F0), 512 , 1, 1)
    F0 = F0.expand(len(F0), 512, 64, 64)
    print("Cat Shape",F0.shape, x.shape)
    x = torch.cat((x, F0), dim = 1)
    x = self.mix1(x)
    x = self.expand1(x, skip3)

    F1 = F0
    F1 = self.conv1(F1)
    x = torch.cat((x, F1), dim = 1)
    x = self.mix2(x)
    x = self.expand2(x, skip2)

    F2 = F1
    F2 = self.conv2(F2)
    x = torch.cat((x, F2), dim = 1)
    x = self.mix3(x)
    x = self.expand3(x, skip1)

    F3 = F2
    F3 = self.conv3(F3)
    # print('Here', F3.shape, x.shape)
    # F3_attention = self.cross(x, F3)
    # print(F3_attention.shape)
    x = torch.cat((x, F3), dim = 1)
    x = self.mix4(x)
    x = self.convf(x)

    return x

In [ ]:
model_mf = UNet_MM_BS(1,1, clip_tokenizer, clip)
out_mf = model_mf(torch.randn(3,1,512,512),['The cone is not properly segmented','The cone is well segmented','The cone is somewhat segmented'])
# print(out_mf.shape)

/usr/local/lib/python3.10/dist-packages/torch/nn/init.py:412: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


torch.Size([3, 64]) torch.Size([3, 128]) torch.Size([3, 256])
torch.Size([3, 512])
Cat Shape torch.Size([3, 512, 64, 64]) torch.Size([3, 256, 64, 64])


In [ ]:
from transformers import AutoProcessor, AutoTokenizer, CLIPModel
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

# Cross-Attention

In [ ]:
class ContractingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ContractingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    skip = x
    x = self.maxpool(x)

    return skip, x


class ExpandingBlock(nn.Module):
  def __init__(self, in_channels, out_channels):
    super(ExpandingBlock, self).__init__()

    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu1 = nn.ReLU()

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.relu2 = nn.ReLU()

    self.upsample = nn.ConvTranspose2d(out_channels, out_channels//2, kernel_size=2, stride=2)

  def forward(self, x, skip):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu1(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu2(x)

    x = self.upsample(x)

    x = torch.cat((x, skip), dim = 1)

    return x


class CrossAttention(nn.Module):
  def __init__(self, in_channels):
    super(CrossAttention, self).__init__()

    self.query = nn.Conv2d(in_channels, in_channels//8, kernel_size=1)
    self.key = nn.Conv2d(in_channels, in_channels//8, kernel_size=1)
    self.value = nn.Conv2d(in_channels, in_channels//8, kernel_size=1)

  def forward(self, x, xt):
    print('Reached Here')
    print(x.shape, xt.shape)
    k = self.key(x)
    print(k.shape)
    v = self.value(x)
    print(v.shape)

    q = self.query(xt)
    print(q.shape)

    print('s1',k.shape,v.shape,q.shape)

    q = q.view(q.size(0), -1, q.size(2) * q.size(3))
    k = k.view(k.size(0), -1, k.size(2) * k.size(3))
    v = v.view(v.size(0), -1, v.size(2) * v.size(3))

    attn_scores = F.softmax(torch.bmm(q.transpose(1, 2), k), dim=2)
    attention = torch.bmm(v, attn_scores.transpose(1, 2))
    attention = attention.view(v.size(0), v.size(1), v.size(2), xt.size(2), xt.size(3))

    return attention


class UNet_MM_BS(nn.Module):
  def __init__(self, in_channels, out_channels, clip_tokenizer, clip):
    super(UNet_MM_BS, self).__init__()

    self.contract1 = ContractingBlock(in_channels, 64)
    self.contract2 = ContractingBlock(64, 128)
    self.contract3 = ContractingBlock(128, 256)

    self.pool1 = nn.AdaptiveAvgPool2d((1,1))
    self.pool2 = nn.AdaptiveAvgPool2d((1,1))
    self.pool3 = nn.AdaptiveAvgPool2d((1,1))

    self.mix1 = nn.Conv2d(768, 256, kernel_size=1)

    self.mix2 = nn.Conv2d(640, 128, kernel_size=1)
    self.mix3 = nn.Conv2d(320, 64, kernel_size=1)
    self.mix4 = nn.Conv2d(160, 64, kernel_size=1)

    self.clip = clip
    self.clip_tokenizer = clip_tokenizer

    self.linear1 = nn.Linear(64, 512)
    self.linear2 = nn.Linear(128, 512)
    self.linear3 = nn.Linear(256, 512)
    self.linear4 = nn.Linear(512, 512)
    self.linear5 = nn.Linear(512, 512)
    self.linear6 = nn.Linear(1024, 512)
    self.linear7 = nn.Linear(1024, 512)
    self.linear8 = nn.Linear(1024, 512)
    self.linear9 = nn.Linear(1536, 512)

    self.conv1 = nn.ConvTranspose2d(in_channels=512, out_channels=320, kernel_size=3, stride=2, padding=1, output_padding=1)
    self.conv2 = nn.ConvTranspose2d(in_channels=320, out_channels=160, kernel_size=3, stride=2, padding=1, output_padding=1)
    self.conv3 = nn.ConvTranspose2d(in_channels=160, out_channels=80, kernel_size=3, stride=2, padding=1, output_padding=1)

    self.expand1 = ExpandingBlock(256, 128)
    self.expand2 = ExpandingBlock(128, 64)
    self.expand3 = ExpandingBlock(64, 32)

    self.cross1 = CrossAttention(320)
    self.cross2 = CrossAttention(160)
    self.cross3 = CrossAttention(80)

    self.convf = nn.Conv2d(64, out_channels, kernel_size = 1)

  def forward(self, x, y):

    skip1, x = self.contract1(x)
    skip2, x = self.contract2(x)
    skip3, x = self.contract3(x)

    fuse1 = skip1
    fuse2 = skip2
    fuse3 = skip3

    fuse1 = self.pool1(fuse1)
    fuse2 = self.pool1(fuse2)
    fuse3 = self.pool1(fuse3)

    vfuse1 = fuse1.reshape(fuse1.shape[0],-1)
    vfuse2 = fuse2.reshape(fuse2.shape[0],-1)
    vfuse3 = fuse3.reshape(fuse3.shape[0],-1)

    print(vfuse1.shape, vfuse2.shape, vfuse3.shape)

    inputs = self.clip_tokenizer(y, padding=True, return_tensors="pt")
    text_features = self.clip.get_text_features(**inputs)
    y = clip.get_text_features(**inputs)
    y = y.reshape(len(y),-1)
    print(y.shape)
    base_y = y

    F3 = F.relu(self.linear3(vfuse3))*F.relu(self.linear4(base_y))
    F2 = torch.cat([F.relu(self.linear2(vfuse2)), F.relu(self.linear5(F3))], dim=1)
    F1 = torch.cat([F.relu(self.linear1(vfuse1)), F.relu(self.linear6(F2))], dim=1)
    F0 = self.linear9(torch.cat([self.linear8(F1),self.linear7(F2),F3],dim=1))

    F0 = F0.reshape(len(F0), 512 , 1, 1)
    F0 = F0.expand(len(F0), 512, 64, 64)
    print("Cat Shape",F0.shape, x.shape)
    x = torch.cat((x, F0), dim = 1)
    x = self.mix1(x)
    x = self.expand1(x, skip3)

    F1 = F0
    F1 = self.conv1(F1)
    x = torch.cat((x, F1), dim = 1)
    x = self.mix2(x)
    x = self.expand2(x, skip2)

    F2 = F1
    F2 = self.conv2(F2)
    x = torch.cat((x, F2), dim = 1)
    x = self.mix3(x)
    x = self.expand3(x, skip1)

    F3 = F2
    F3 = self.conv3(F3)
    print('Here', F3.shape, x.shape)
    F3_attention = self.cross3(x, F3)
    print(F3_attention.shape)
    x = torch.cat((x, F3), dim = 1)
    x = self.mix4(x)
    x = self.convf(x)

    return x

In [ ]:
model_mf = UNet_MM_BS(1,1, clip_tokenizer, clip)
out_mf = model_mf(torch.randn(3,1,512,512),['The cone is not properly segmented','The cone is well segmented','The cone is somewhat segmented'])
# print(out_mf.shape)

torch.Size([3, 64]) torch.Size([3, 128]) torch.Size([3, 256])
torch.Size([3, 512])
Cat Shape torch.Size([3, 512, 64, 64]) torch.Size([3, 256, 64, 64])
Here torch.Size([3, 80, 512, 512]) torch.Size([3, 80, 512, 512])
Reached Here
torch.Size([3, 80, 512, 512]) torch.Size([3, 80, 512, 512])
torch.Size([3, 10, 512, 512])
torch.Size([3, 10, 512, 512])
torch.Size([3, 10, 512, 512])
s1 torch.Size([3, 10, 512, 512]) torch.Size([3, 10, 512, 512]) torch.Size([3, 10, 512, 512])


In [ ]:
from transformers import AutoProcessor, AutoTokenizer, CLIPModel
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

# Test

In [ ]:
import torch
import torch.nn as nn

# Assuming you have a tensor with size [3, 512, 64, 64]
input_tensor = torch.randn(3, 512, 64, 64)

# Define the transposed convolutional layer
conv_transpose_layer = nn.ConvTranspose2d(in_channels=512, out_channels=320, kernel_size=3, stride=2, padding=1, output_padding=1)

# Apply the transposed convolutional layer to upsample
upsampled_tensor = conv_transpose_layer(input_tensor)

# Check the size after upsampling
print("Upsampled Size:", upsampled_tensor.size())

# Now you can apply a regular convolution to adjust the channels
final_conv_layer = nn.Conv2d(in_channels=320, out_channels=320, kernel_size=3, stride=1, padding=1)
output_tensor = final_conv_layer(upsampled_tensor)

# Print the sizes
print("Input Size:", input_tensor.size())
print("Output Size:", output_tensor.size())


Upsampled Size: torch.Size([3, 320, 128, 128])
Input Size: torch.Size([3, 512, 64, 64])
Output Size: torch.Size([3, 320, 128, 128])


In [ ]:
# import torch
# from transformers import DistilBertModel, DistilBertTokenizer

# # Load pre-trained DistilBERT model and tokenizer
# model_name = "distilbert-base-uncased"
# tokenizer = DistilBertTokenizer.from_pretrained(model_name)
# model = DistilBertModel.from_pretrained(model_name)

# # Assuming 'y' is a list of sentences
# y = ["Sentence 1wwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwww", "Sentence 2", "Sentence 3"]

# # Tokenize and obtain embeddings for each sentence
# embeddings = []

# for sentence in y:
#     # Tokenize text
#     inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)

#     # Get embeddings
#     with torch.no_grad():
#         outputs = model(**inputs)
#         last_hidden_states = outputs.last_hidden_state

#     # Perform mean pooling
#     mean_pooled = last_hidden_states.mean(dim=1)

#     # Append the mean_pooled embedding to the list
#     embeddings.append(mean_pooled)

# # Stack the embeddings into a single tensor
# embeddings = torch.stack(embeddings)

# # Now, embeddings is a tensor containing constant-sized embeddings for each sentence
# print(embeddings.shape)
